Baseline Modeling

This notebook builds baseline predictive models for customer churn, evaluates their performance, and interprets results.


Importing Libraries

In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

Load Processed Data

In [23]:

df = pd.read_csv("../data/processed/Telco-Customer-Churn-processed.csv")
df.head()


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


Split features & target

In [24]:
X = df.drop('Churn', axis=1)
y = df['Churn']

Train-Test Split (80/20) with stratification

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Scale numeric features

In [26]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']  # adjust if you have more numeric features
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

Baseline Logistic Regression

In [27]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

Model Evaluation

In [28]:
y_pred = log_reg.predict(X_test)
y_pred_proba = log_reg.predict_proba(X_test)[:, 1]
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

ROC-AUC: 0.8420677361853834


Inspect coefficients

In [29]:
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': log_reg.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print("\nTop 10 positive coefficients:\n", coef_df.head(10))
print("\nTop 10 negative coefficients:\n", coef_df.tail(10))


Top 10 positive coefficients:
                            Feature  Coefficient
10     InternetService_Fiber optic     1.190576
3                     TotalCharges     0.511378
23             StreamingMovies_Yes     0.380384
21                 StreamingTV_Yes     0.378872
28  PaymentMethod_Electronic check     0.377641
26            PaperlessBilling_Yes     0.371600
9                MultipleLines_Yes     0.365301
0                    SeniorCitizen     0.144638
29      PaymentMethod_Mailed check     0.067582
17            DeviceProtection_Yes     0.037562

Top 10 negative coefficients:
                                Feature  Coefficient
12  OnlineSecurity_No internet service    -0.174340
6                       Dependents_Yes    -0.225854
8       MultipleLines_No phone service    -0.246604
19                     TechSupport_Yes    -0.298012
13                  OnlineSecurity_Yes    -0.347516
2                       MonthlyCharges    -0.477810
7                     PhoneService_Yes    -0